# Label Smoothing

**Goal:** Implement label-smoothed cross-entropy from scratch, validate it against `F.cross_entropy(label_smoothing=eps)`, then show how smoothing changes training loss, model confidence, and calibration-like metrics compared to hard-label training.

Cross-links: `[[cross-entropy-nll]]`, `[[softmax]]`, `[[evaluation-metrics]]`

## Configuration

Device, random seed, and default dtype come from `shared.config.configure()`. This reads `config.toml` at the repo root and applies the chosen device / seed / dtype for the session.

In [1]:
import sys
from pathlib import Path

import matplotlib

matplotlib.use("Agg")  # noqa
import matplotlib.pyplot as plt  # noqa: E402
import torch  # noqa: E402
import torch.nn.functional as F  # noqa: E402


def _find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists():
            return p
    return start


REPO_ROOT = _find_repo_root(Path.cwd())
sys.path.insert(0, str(REPO_ROOT))

from shared.config import configure  # noqa: E402

device = configure()
print("running on:", device)

running on: mps


## From Scratch: Smoothed Target Construction

For `K` classes and smoothing parameter `epsilon`, we replace the hard one-hot target with:

```text
y_smooth = (1 - epsilon) * one_hot + epsilon / K
```

So the correct class receives probability `1 - epsilon + epsilon/K` and every incorrect class receives `epsilon/K`. The gradient of cross-entropy w.r.t. logits changes from `p - e_y` (hard) to `p - q` (smoothed), where the gradient is never exactly `-1` for the correct class — the model is never pushed to assign probability 1.

This is a loss-target modification, not a change to the model architecture.

In [2]:
def smooth_one_hot(
    y: torch.Tensor,
    num_classes: int,
    eps: float,
) -> torch.Tensor:
    """Construct label-smoothed target distributions.

    Args:
        y: Integer class labels, shape (B,).
        num_classes: Number of classes K.
        eps: Smoothing parameter in [0, 1).

    Returns:
        Soft target distribution, shape (B, K).
    """
    # One-hot: shape (B, K)
    one_hot = torch.zeros(y.shape[0], num_classes, device=y.device)
    one_hot.scatter_(1, y.unsqueeze(1), 1.0)
    return (1.0 - eps) * one_hot + eps / num_classes


# Illustrative example: K=5, correct class=2, eps=0.1
K = 5
y_ex = torch.tensor([2], device=device)
eps_ex = 0.1
q_ex = smooth_one_hot(y_ex, K, eps_ex)
print("Hard one-hot:", torch.zeros(K).scatter_(0, torch.tensor(2), 1.0).tolist())
print("Smoothed:    ", q_ex[0].tolist())
print(f"  Correct class target: {q_ex[0, 2].item():.4f}  (expected {1 - eps_ex + eps_ex/K:.4f})")
print(f"  Each wrong class:     {q_ex[0, 0].item():.4f}  (expected {eps_ex/K:.4f})")
assert abs(q_ex[0].sum().item() - 1.0) < 1e-6, "Target probabilities must sum to 1"
print("  Probabilities sum to 1.0 ✓")

Hard one-hot: [0.0, 0.0, 1.0, 0.0, 0.0]
Smoothed:     [0.019999999552965164, 0.019999999552965164, 0.9199999570846558, 0.019999999552965164, 0.019999999552965164]
  Correct class target: 0.9200  (expected 0.9200)
  Each wrong class:     0.0200  (expected 0.0200)
  Probabilities sum to 1.0 ✓


## From Scratch: Soft-Target Cross-Entropy

Standard cross-entropy with soft targets `q` and model log-probabilities `log(p)`:

```text
L = -sum_k q_k * log(p_k)
  = -sum_k q_k * (z_k - log(sum_j exp(z_j)))
```

With a log-sum-exp trick for numerical stability. In vector form across a batch this is:

```text
L_batch = mean over i of [-sum_k q_{ik} * log_softmax(z_i)_k]
        = mean(-q * log_softmax(z))  # element-wise, then sum over classes, then mean over batch
```

In [3]:
def label_smoothing_cross_entropy(
    logits: torch.Tensor,
    y: torch.Tensor,
    eps: float,
) -> torch.Tensor:
    """Label-smoothed cross-entropy loss from scratch.

    Args:
        logits: Raw model outputs, shape (B, K).
        y: Integer class labels, shape (B,).
        eps: Label smoothing parameter.

    Returns:
        Scalar mean loss.
    """
    K = logits.shape[1]
    q = smooth_one_hot(y, K, eps)                    # (B, K)
    log_p = F.log_softmax(logits, dim=1)             # (B, K) — numerically stable
    # Soft CE: -sum_k q_k * log_p_k, averaged over batch
    per_sample_loss = -(q * log_p).sum(dim=1)        # (B,)
    return per_sample_loss.mean()


# Quick numerical sanity check
torch.manual_seed(0)
B_chk, K_chk = 8, 4
logits_chk = torch.randn(B_chk, K_chk, device=device)
y_chk = torch.randint(0, K_chk, (B_chk,), device=device)

# eps=0 should match hard-label cross-entropy exactly
loss_eps0 = label_smoothing_cross_entropy(logits_chk, y_chk, eps=0.0)
loss_hard  = F.cross_entropy(logits_chk, y_chk)
print(f"eps=0 scratch: {loss_eps0.item():.6f}")
print(f"F.cross_entropy: {loss_hard.item():.6f}")
assert abs(loss_eps0.item() - loss_hard.item()) < 1e-5, "eps=0 should equal hard CE"
print("PASSED: eps=0 matches F.cross_entropy")

eps=0 scratch: 1.636530
F.cross_entropy: 1.636530
PASSED: eps=0 matches F.cross_entropy


## Validation: Assert Against `F.cross_entropy(label_smoothing=eps)`

PyTorch 1.10+ supports `F.cross_entropy(..., label_smoothing=eps)` natively. We validate our scratch implementation against this reference across several values of `eps`.

In [4]:
torch.manual_seed(42)
B_v, K_v = 64, 10
logits_v = torch.randn(B_v, K_v, device=device)
y_v = torch.randint(0, K_v, (B_v,), device=device)

eps_values = [0.0, 0.05, 0.1, 0.2, 0.5]
print(f"{'eps':>6}  {'scratch':>12}  {'torch':>12}  {'abs diff':>12}  {'status':>8}")
print("-" * 60)
for eps in eps_values:
    loss_scratch = label_smoothing_cross_entropy(logits_v, y_v, eps)
    loss_torch   = F.cross_entropy(logits_v, y_v, label_smoothing=eps)
    diff = abs(loss_scratch.item() - loss_torch.item())
    status = "PASS" if diff < 1e-5 else "FAIL"
    print(f"{eps:>6.2f}  {loss_scratch.item():>12.6f}  {loss_torch.item():>12.6f}  {diff:>12.2e}  {status:>8}")
    assert diff < 1e-5, f"Mismatch at eps={eps}: {diff}"

print("\nAll eps values: scratch matches F.cross_entropy ✓")

   eps       scratch         torch      abs diff    status
------------------------------------------------------------
  0.00      2.634538      2.634538      0.00e+00      PASS
  0.05      2.638066      2.638065      2.38e-07      PASS
  0.10      2.641593      2.641593      0.00e+00      PASS
  0.20      2.648649      2.648649      0.00e+00      PASS
  0.50      2.669816      2.669816      0.00e+00      PASS

All eps values: scratch matches F.cross_entropy ✓


## Effect on Loss and Model Confidence

When we train with hard labels, the model learns to push logits toward ±∞ to assign probability 1 to the correct class. Label smoothing places a small mass on all classes, creating a gradient that resists this infinite logit gap.

We train identical small classifiers with different `eps` values on synthetic data and compare:
- Training and validation cross-entropy (NLL)
- Average **max softmax probability** (a proxy for overconfidence)

High max softmax probability ≈ overconfident; lower values suggest the model is less extreme in its predictions.

In [5]:
import torch.nn as nn


def make_synthetic_data(
    n: int = 1500, d: int = 10, K: int = 4, noise: float = 0.3
) -> tuple[torch.Tensor, torch.Tensor]:
    """Generate a synthetic K-class dataset from noisy linear scores."""
    torch.manual_seed(0)
    W = torch.randn(d, K)                            # class centroids
    X = torch.randn(n, d)
    scores = X @ W + noise * torch.randn(n, K)
    y = scores.argmax(dim=1)
    return X.to(device), y.to(device)


X_all, y_all = make_synthetic_data()
n_train = 1200
X_train, y_train = X_all[:n_train], y_all[:n_train]
X_val,   y_val   = X_all[n_train:], y_all[n_train:]


class Classifier(nn.Module):
    def __init__(self, d: int = 10, K: int = 4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d, 64), nn.ReLU(),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, K),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


def train_classifier(eps: float, steps: int = 300) -> dict:
    """Train a classifier with a given label-smoothing value."""
    torch.manual_seed(0)
    model = Classifier().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=3e-3)
    batch_size = 64
    train_losses = []

    for step in range(steps):
        idx = torch.randint(0, n_train, (batch_size,), device=device)
        xb, yb = X_train[idx], y_train[idx]
        logits = model(xb)
        loss = label_smoothing_cross_entropy(logits, yb, eps)
        opt.zero_grad()
        loss.backward()
        opt.step()
        train_losses.append(loss.item())

    model.eval()
    with torch.no_grad():
        val_logits = model(X_val)
        # Always measure val NLL with hard CE (eps=0) to compare apples-to-apples
        val_nll    = F.cross_entropy(val_logits, y_val).item()
        val_acc    = (val_logits.argmax(1) == y_val).float().mean().item()
        probs      = val_logits.softmax(dim=1)
        avg_max_prob = probs.max(dim=1).values.mean().item()

    return {
        "eps": eps,
        "val_nll": val_nll,
        "val_acc": val_acc,
        "avg_max_prob": avg_max_prob,
        "train_losses": train_losses,
    }


eps_sweep = [0.0, 0.05, 0.1, 0.2]
results = [train_classifier(eps) for eps in eps_sweep]

print(f"{'eps':>5}  {'val_NLL':>10}  {'val_acc':>10}  {'avg max_prob':>14}")
print("-" * 50)
for r in results:
    print(f"{r['eps']:>5.2f}  {r['val_nll']:>10.4f}  {r['val_acc']:>10.4f}  {r['avg_max_prob']:>14.4f}")

  eps     val_NLL     val_acc    avg max_prob
--------------------------------------------------
 0.00      0.1229      0.9567          0.9471
 0.05      0.1940      0.9467          0.8728
 0.10      0.2541      0.9400          0.8234
 0.20      0.3570      0.9367          0.7416


In [6]:
# Plot training curves and confidence comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Training loss curves
ax = axes[0]
for r in results:
    ax.plot(r["train_losses"], label=f"eps={r['eps']}", alpha=0.8)
ax.set_xlabel("Step")
ax.set_ylabel("Training loss (smoothed CE)")
ax.set_title("Training Loss by Smoothing Value")
ax.legend()

# Confidence comparison
ax = axes[1]
eps_vals = [r["eps"] for r in results]
max_probs = [r["avg_max_prob"] for r in results]
val_nlls  = [r["val_nll"] for r in results]

color1, color2 = "steelblue", "tomato"
ax2 = ax.twinx()
bars = ax.bar(eps_vals, max_probs, width=0.03, color=color1, alpha=0.7, label="Avg max prob (L)")
ax2.plot(eps_vals, val_nlls, "o-", color=color2, label="Val NLL (R)")
ax.set_xlabel("Label smoothing eps")
ax.set_ylabel("Avg max softmax probability", color=color1)
ax2.set_ylabel("Validation NLL", color=color2)
ax.set_title("Confidence vs. NLL by Smoothing Value")
ax.set_ylim(0, 1)

lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc="upper right")

fig.tight_layout()
plt.savefig("label_smoothing_comparison.png", dpi=80)
plt.close(fig)
print("Saved label_smoothing_comparison.png")

# Verify smoothing reduces confidence
assert results[-1]["avg_max_prob"] < results[0]["avg_max_prob"], (
    "Expected label smoothing to reduce average max softmax probability"
)
print("PASSED: avg max probability decreases with larger eps")
print(f"  eps=0.0: {results[0]['avg_max_prob']:.4f}")
print(f"  eps=0.2: {results[-1]['avg_max_prob']:.4f}")

Saved label_smoothing_comparison.png
PASSED: avg max probability decreases with larger eps
  eps=0.0: 0.9471
  eps=0.2: 0.7416


## Gradient Analysis: Hard vs. Smoothed Targets

The gradient of cross-entropy w.r.t. logit `z_k` is `p_k - q_k`. With hard labels `q = e_y`:

- Correct class: `p_y - 1` (always negative, pushes logit up)
- Wrong class: `p_k - 0 = p_k` (always positive, pushes logit down)

With smoothed labels `q_k = eps/K` for wrong classes and `q_y = 1 - eps + eps/K` for correct:

- Correct class: `p_y - (1 - eps + eps/K)` — smaller magnitude, especially when `p_y` is large
- Wrong class: `p_k - eps/K` — gradient becomes zero when `p_k = eps/K`, not zero

The model is discouraged from driving `p_y -> 1` because the gradient towards 1 is weaker. This is why smoothing **reduces overconfidence** but does **not guarantee calibration** — the model may still be miscalibrated, just less extremely so.

In [7]:
# Demonstrate gradient differences on synthetic logits
torch.manual_seed(3)
K_grad = 4
z = torch.randn(1, K_grad, device=device, requires_grad=True)
y_grad = torch.tensor([0], device=device)

p = z.softmax(dim=1)
print(f"Logits:       {z.detach().squeeze().tolist()}")
print(f"Softmax p:    {p.detach().squeeze().tolist()}")
print()

for eps_g in [0.0, 0.1, 0.2]:
    if z.grad is not None:
        z.grad.zero_()
    loss_g = label_smoothing_cross_entropy(z, y_grad, eps=eps_g)
    loss_g.backward()
    grad = z.grad.clone().squeeze()
    # Gradient for correct class (index 0) = p_0 - q_0
    q0 = 1.0 - eps_g + eps_g / K_grad
    expected_grad0 = p[0, 0].item() - q0
    print(f"eps={eps_g:.1f}: grad = {grad.tolist()}")
    print(f"        expected grad[0] = p[0]-q[0] = {p[0,0].item():.4f} - {q0:.4f} = {expected_grad0:.4f}  actual={grad[0].item():.4f}")
    assert abs(grad[0].item() - expected_grad0) < 1e-5
print("\nPASSED: gradient = p - q for all eps values")

Logits:       [0.22207623720169067, -0.9872624278068542, -0.6002779603004456, 0.6860417723655701]
Softmax p:    [0.3004664182662964, 0.08965754508972168, 0.13202375173568726, 0.4778522253036499]

eps=0.0: grad = [-0.6995335817337036, 0.08965754508972168, 0.13202372193336487, 0.4778522551059723]
        expected grad[0] = p[0]-q[0] = 0.3005 - 1.0000 = -0.6995  actual=-0.6995
eps=0.1: grad = [-0.6245335340499878, 0.0646575391292572, 0.1070237085223198, 0.4528522193431854]
        expected grad[0] = p[0]-q[0] = 0.3005 - 0.9250 = -0.6245  actual=-0.6245
eps=0.2: grad = [-0.5495336055755615, 0.03965754434466362, 0.0820237249135971, 0.42785224318504333]
        expected grad[0] = p[0]-q[0] = 0.3005 - 0.8500 = -0.5495  actual=-0.5495

PASSED: gradient = p - q for all eps values


In [8]:
# Simple calibration table: predicted confidence vs. actual accuracy
def calibration_table(logits: torch.Tensor, y: torch.Tensor, n_bins: int = 5) -> None:
    """Print a simple reliability table: bucket by max prob, show accuracy per bucket."""
    probs = logits.softmax(dim=1)
    max_probs, preds = probs.max(dim=1)
    correct = (preds == y)
    edges = torch.linspace(0, 1, n_bins + 1)
    print(f"{'Confidence bucket':>22}  {'count':>6}  {'accuracy':>10}  {'avg conf':>10}")
    print("-" * 58)
    for lo, hi in zip(edges[:-1], edges[1:]):
        mask = (max_probs >= lo) & (max_probs < hi)
        n = mask.sum().item()
        if n == 0:
            continue
        acc  = correct[mask].float().mean().item()
        conf = max_probs[mask].mean().item()
        print(f"  [{lo:.1f}, {hi:.1f})  {n:>6}  {acc:>10.3f}  {conf:>10.3f}")


# Re-run to get models with eps=0 and eps=0.1
torch.manual_seed(0)
res_hard = train_classifier(0.0)
res_smooth = train_classifier(0.1)

print("=== Hard Labels (eps=0.0) ===")
model_hard = Classifier().to(device)
# (Re-train to get the actual model state)
torch.manual_seed(0)
model_hard2 = Classifier().to(device)
opt2 = torch.optim.Adam(model_hard2.parameters(), lr=3e-3)
for step in range(300):
    idx = torch.randint(0, n_train, (64,), device=device)
    xb, yb = X_train[idx], y_train[idx]
    loss = label_smoothing_cross_entropy(model_hard2(xb), yb, 0.0)
    opt2.zero_grad(); loss.backward(); opt2.step()
model_hard2.eval()
with torch.no_grad():
    logits_hard_val = model_hard2(X_val)
calibration_table(logits_hard_val, y_val)

print("\n=== Label Smoothing (eps=0.1) ===")
torch.manual_seed(0)
model_smooth2 = Classifier().to(device)
opt3 = torch.optim.Adam(model_smooth2.parameters(), lr=3e-3)
for step in range(300):
    idx = torch.randint(0, n_train, (64,), device=device)
    xb, yb = X_train[idx], y_train[idx]
    loss = label_smoothing_cross_entropy(model_smooth2(xb), yb, 0.1)
    opt3.zero_grad(); loss.backward(); opt3.step()
model_smooth2.eval()
with torch.no_grad():
    logits_smooth_val = model_smooth2(X_val)
calibration_table(logits_smooth_val, y_val)
print("\nNote: lower avg confidence with smoothing does not guarantee perfect calibration,")
print("but it often moves the model closer to the true accuracy per confidence bucket.")

=== Hard Labels (eps=0.0) ===
     Confidence bucket   count    accuracy    avg conf
----------------------------------------------------------


  [0.4, 0.6)      12       0.583       0.527
  [0.6, 0.8)      16       0.938       0.686
  [0.8, 1.0)     243       0.971       0.979

=== Label Smoothing (eps=0.1) ===


     Confidence bucket   count    accuracy    avg conf
----------------------------------------------------------
  [0.2, 0.4)       3       0.333       0.371
  [0.4, 0.6)      37       0.757       0.508
  [0.6, 0.8)      61       0.902       0.701
  [0.8, 1.0)     199       0.995       0.926

Note: lower avg confidence with smoothing does not guarantee perfect calibration,
but it often moves the model closer to the true accuracy per confidence bucket.


## Idiomatic PyTorch: `nn.CrossEntropyLoss(label_smoothing=eps)`

In production code, label smoothing is a single parameter on the loss constructor. There is no change to the model architecture — only the training target changes. The inference path (softmax, argmax, model logits) is identical with or without smoothing.

In [9]:
# Standard production idiom
torch.manual_seed(0)
model_prod = Classifier().to(device)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)   # that's all it takes
opt_prod = torch.optim.Adam(model_prod.parameters(), lr=3e-3)

for step in range(300):
    idx = torch.randint(0, n_train, (64,), device=device)
    xb, yb = X_train[idx], y_train[idx]
    loss = criterion(model_prod(xb), yb)
    opt_prod.zero_grad()
    loss.backward()
    opt_prod.step()

model_prod.eval()
with torch.no_grad():
    logits_prod = model_prod(X_val)
    acc_prod = (logits_prod.argmax(1) == y_val).float().mean().item()
    avg_max_prod = logits_prod.softmax(dim=1).max(1).values.mean().item()

print(f"nn.CrossEntropyLoss(label_smoothing=0.1)")
print(f"  Validation accuracy:          {acc_prod:.4f}")
print(f"  Average max softmax prob:     {avg_max_prod:.4f}")

# Confirm it matches our scratch implementation (same seed, same eps)
torch.manual_seed(0)
model_ref2 = Classifier().to(device)
opt_ref2 = torch.optim.Adam(model_ref2.parameters(), lr=3e-3)
for step in range(300):
    idx = torch.randint(0, n_train, (64,), device=device)
    xb, yb = X_train[idx], y_train[idx]
    loss = label_smoothing_cross_entropy(model_ref2(xb), yb, 0.1)
    opt_ref2.zero_grad(); loss.backward(); opt_ref2.step()
model_ref2.eval()
with torch.no_grad():
    logits_ref2 = model_ref2(X_val)
    acc_ref2 = (logits_ref2.argmax(1) == y_val).float().mean().item()

print(f"\nScratch implementation (same seed, same eps=0.1)")
print(f"  Validation accuracy:          {acc_ref2:.4f}")
# Accuracy should match closely (same random seed, same training procedure)
print(f"\nAccuracy difference: {abs(acc_prod - acc_ref2):.4f}")

nn.CrossEntropyLoss(label_smoothing=0.1)
  Validation accuracy:          0.9400
  Average max softmax prob:     0.8234

Scratch implementation (same seed, same eps=0.1)
  Validation accuracy:          0.9400

Accuracy difference: 0.0000


## Takeaways

- **Smoothed target:** `y_smooth = (1 - eps) * one_hot + eps / K`. Correct class receives `1 - eps + eps/K`; every class (including wrong ones) receives `eps/K`.
- **Soft CE loss:** `L = -sum_k q_k * log(p_k)`. With `eps=0` this reduces exactly to standard cross-entropy.
- **Gradient:** `dL/dz_k = p_k - q_k`. The gradient for the correct class is never `-1` — the model cannot be trained to assign probability 1.
- **Reduces overconfidence:** average max softmax probability decreases as `eps` increases. This is not the same as calibration, but it moves predictions in the right direction.
- **Calibration caveat:** smoothing does not guarantee that "70% confident" predictions are correct 70% of the time — it only reduces the *magnitude* of overconfidence. Temperature scaling is a better tool for calibration after training.
- **When to avoid:** knowledge distillation (teacher soft labels already carry information — smoothing corrupts them), tasks with very rare classes (smoothing weakens an already sparse signal), and when you need accurate probability estimates.
- **Idiomatic use:** `nn.CrossEntropyLoss(label_smoothing=eps)` — no model changes required, only the training loss changes. Cross-links: `[[cross-entropy-nll]]`, `[[softmax]]`.